## A6d - Build solar-to-L1 targets

This notebook creates supervised targets for the Solar-to-L1 bridge model.

### Input
- `Data/processed/master_3h.parquet` (L1 solar-wind table)

### Targets (lead times)
Horizon set: 24h, 36h, 48h, 96h

Target columns:
- `bz_gsm_3h_min`
- `sw_speed_3h_mean`
- `sw_density_3h_mean`

### Output
- `Data/processed/solar_to_l1_targets.parquet`


In [6]:
import pandas as pd
from pathlib import Path

L1_PATH = Path('../../Data/processed/master_3h.parquet')
OUT_PATH = Path('../../Data/processed/solar_to_l1_targets.parquet')

HORIZONS_H = [24, 36, 48, 96]

if not L1_PATH.exists():
    raise FileNotFoundError(f'Cannot find {L1_PATH}')

df = pd.read_parquet(L1_PATH)
# ensure timestamp
if 'timestamp_utc' not in df.columns:
    # fallback: try common names
    for alt in ['timestamp', 'time', 'datetime']:
        if alt in df.columns:
            df = df.rename(columns={alt: 'timestamp_utc'})
            break

df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'], utc=True)
df = df.sort_values('timestamp_utc')

# select base L1 columns
needed = ['bz_gsm_3h_min', 'sw_speed_3h_mean', 'sw_density_3h_mean']
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns in L1 data: {missing}')

targets = []
for h in HORIZONS_H:
    shift_steps = h // 3  # 3h grid
    tmp = df[['timestamp_utc']].copy()
    tmp['horizon_h'] = h
    tmp['bz_gsm_3h_min'] = df['bz_gsm_3h_min'].shift(-shift_steps)
    tmp['sw_speed_3h_mean'] = df['sw_speed_3h_mean'].shift(-shift_steps)
    tmp['sw_density_3h_mean'] = df['sw_density_3h_mean'].shift(-shift_steps)
    targets.append(tmp)

out = pd.concat(targets, ignore_index=True)
out = out.dropna(subset=['bz_gsm_3h_min', 'sw_speed_3h_mean', 'sw_density_3h_mean'])

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
out.to_parquet(OUT_PATH, index=False)

print(f'Saved {len(out)} rows -> {OUT_PATH}')
print(out.head())



Saved 524270 rows -> ../../Data/processed/solar_to_l1_targets.parquet
              timestamp_utc  horizon_h  bz_gsm_3h_min  sw_speed_3h_mean  \
0 1963-11-28 00:00:00+00:00         24            0.4        284.333333   
1 1963-11-28 03:00:00+00:00         24           -1.0        290.333333   
2 1963-11-28 06:00:00+00:00         24           -0.7        286.666667   
3 1963-11-28 09:00:00+00:00         24           -0.3        290.000000   
4 1963-11-28 12:00:00+00:00         24           -1.8        291.000000   

   sw_density_3h_mean  
0           11.833333  
1           17.500000  
2           21.833333  
3           28.833333  
4           39.333333  


In [5]:
df.columns

Index(['timestamp_utc', 'kp', 'bz_gsm_3h_min', 'by_gsm_3h_mean',
       'b_scalar_3h_mean', 'sw_speed_3h_mean', 'sw_density_3h_mean',
       'sw_temperature_3h_mean'],
      dtype='object')